In [1]:
# Install required packages
import subprocess
import sys

# def install_package(package):
#     try:
#         subprocess.check_call([sys.executable, "-m", "pip", "install", package])
#         print(f"Successfully installed {package}")
#     except subprocess.CalledProcessError as e:
#         print(f"Failed to install {package}: {e}")

# # Install packages
# packages = ["h2o", "pandas", "numpy", "scikit-learn", "matplotlib", "seaborn"]
# for package in packages:
#     install_package(package)

# H2O Regression Pipeline for Flight Time Prediction

This notebook implements a regression pipeline using H2O AutoML to predict actual departure and arrival times for flights. Unlike the classification version that predicts delays, this version focuses on predicting the exact timing of flights.

## Key Differences from Classification Pipeline:
- **Target Variables**: Actual departure time (ATD) and actual arrival time (ATA)
- **Problem Type**: Multi-target regression instead of binary classification
- **Evaluation Metrics**: MAE, RMSE, R² instead of AUC, F1-score
- **Business Value**: Precise time predictions for operational planning

In [2]:
# Import required libraries
import h2o
from h2o.automl import H2OAutoML
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings

warnings.filterwarnings("ignore")

# Initialize H2O
print("Initializing H2O...")
h2o.init(max_mem_size="20G")
print("H2O initialized successfully!")

Initializing H2O...
Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "21.0.8" 2025-07-15; OpenJDK Runtime Environment (build 21.0.8+9-Ubuntu-0ubuntu124.04.1); OpenJDK 64-Bit Server VM (build 21.0.8+9-Ubuntu-0ubuntu124.04.1, mixed mode, sharing)
  Starting server from /lhome/kristsey/AvinorHackathon/.venv/lib/python3.12/site-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmpfi9aagv6
  JVM stdout: /tmp/tmpfi9aagv6/h2o_kristsey_started_from_python.out
  JVM stderr: /tmp/tmpfi9aagv6/h2o_kristsey_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.
Please download and install the latest version from: https://h2o-release.s3.amazonaws.com/h2o/latest_stable.html


H2O_cluster_uptime:,01 secs
H2O_cluster_timezone:,Europe/Oslo
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.7
H2O_cluster_version_age:,5 months and 29 days
H2O_cluster_name:,H2O_from_python_kristsey_21lhhn
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,19.98 Gb
H2O_cluster_total_cores:,32
H2O_cluster_allowed_cores:,32
H2O_cluster_status:,"locked, healthy"


H2O initialized successfully!


In [3]:
# Load the historical flights dataset
print("Loading historical flights dataset...")
data_path = "/home/kristsey/AvinorHackathon/data/avinor_data/historical_flights.csv"

# Load data
df = pd.read_csv(data_path)
print(f"Dataset shape: {df.shape}")
print("\nDataset Info:")
print(df.info())
print("\nFirst few rows:")
print(df.head())

Loading historical flights dataset...
Dataset shape: (410442, 11)

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 410442 entries, 0 to 410441
Data columns (total 11 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   flight_id          410442 non-null  object
 1   dep_airport        410442 non-null  object
 2   dep_airport_group  232968 non-null  object
 3   arr_airport        410442 non-null  object
 4   arr_airport_group  231669 non-null  object
 5   service_type       410336 non-null  object
 6   std                410442 non-null  object
 7   sta                410442 non-null  object
 8   cancelled          410442 non-null  int64 
 9   atd                394327 non-null  object
 10  ata                393576 non-null  object
dtypes: int64(1), object(10)
memory usage: 34.4+ MB
None

First few rows:
  flight_id dep_airport dep_airport_group arr_airport arr_airport_group  \
0     WF149         HOV            

In [4]:
# Data Preprocessing for Regression
print("Starting data preprocessing for regression...")

import sys
import os

print(len(df))

# Add the parent directory to the Python path so we can import from src
parent_dir = os.path.dirname(os.path.dirname(os.path.abspath("avinorhackathon")))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

print(sys.path)

# Now we can import from src
from src.preprocessing import DataPreprocessor

data_preprocessor = DataPreprocessor()
df_processed = data_preprocessor.process_hist_flights(df)
print(df_processed["flight_id"].nunique())

print(len(df_processed))


# Remove rows with missing actual departure or arrival times
df_processed = df_processed.dropna(subset=["atd", "ata"])
print(f"Dataset shape after removing missing actual times: {df_processed.shape}")


# Function to convert datetime to minutes since midnight
def datetime_to_minutes(dt):
    """Convert datetime to minutes since midnight"""
    if pd.isna(dt):
        return None
    return dt.hour * 60 + dt.minute + dt.second / 60


# Create regression targets - convert times to minutes since midnight
print("Converting times to regression targets...")

# Scheduled times as features
df_processed["scheduled_dep_minutes"] = df_processed["std"].apply(datetime_to_minutes)
df_processed["scheduled_arr_minutes"] = df_processed["sta"].apply(datetime_to_minutes)

# Actual times as targets
df_processed["actual_dep_minutes"] = df_processed["atd"].apply(datetime_to_minutes)
df_processed["actual_arr_minutes"] = df_processed["ata"].apply(datetime_to_minutes)

# Calculate scheduled duration
df_processed["scheduled_duration"] = (
    df_processed["scheduled_arr_minutes"] - df_processed["scheduled_dep_minutes"]
)

# Handle negative durations (overnight flights)
df_processed.loc[df_processed["scheduled_duration"] < 0, "scheduled_duration"] += (
    24 * 60
)

# Create target variable: deviation from scheduled times in minutes (at least 15 minutes)
df_processed["dep_delay"] = (
    df_processed["actual_dep_minutes"] - df_processed["scheduled_dep_minutes"]
)
df_processed["arr_delay"] = (
    df_processed["actual_arr_minutes"] - df_processed["scheduled_arr_minutes"]
)
# Filter out delays less than 15 minutes

noise_threshold = 0  # Set to 0 for testing

df_processed["dep_delay"] = df_processed["dep_delay"].where(
    df_processed["dep_delay"].abs() >= noise_threshold, 0
)
df_processed["arr_delay"] = df_processed["arr_delay"].where(
    df_processed["arr_delay"].abs() >= noise_threshold, 0
)

# Remove rows with invalid target values
print(f"Dataset shape after creating regression targets: {df_processed.shape}")


# Display target variable statistics
print(f"\nRegression Target Statistics:")
print(
    f"Departure Delay - Mean: {df_processed['dep_delay'].mean():.1f}, Std: {df_processed['dep_delay'].std():.1f}"
)
print(
    f"Arrival Delay - Mean: {df_processed['arr_delay'].mean():.1f}, Std: {df_processed['arr_delay'].std():.1f}"
)

Starting data preprocessing for regression...
410442
['/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/lhome/kristsey/AvinorHackathon/.venv/lib/python3.12/site-packages', '/lhome/kristsey/AvinorHackathon']


Missing values after processing:
flight_id               0
dep_airport             0
dep_airport_group       0
arr_airport             0
arr_airport_group       0
service_type            0
std                     0
sta                     0
atd                  5285
ata                  5882
dtype: int64
dropping rows with missing actual times
Final missing values:
flight_id            0
dep_airport          0
dep_airport_group    0
arr_airport          0
arr_airport_group    0
service_type         0
std                  0
sta                  0
atd                  0
ata                  0
dtype: int64
1234
390856
Dataset shape after removing missing actual times: (390856, 10)
Converting times to regression targets...
Dataset shape after creating regression targets: (390856, 17)

Regression Target Statistics:
Departure Delay - Mean: 2.3, Std: 28.2
Arrival Delay - Mean: 0.5, Std: 49.8


In [5]:
# Feature Engineering for Regression
print("Starting feature engineering for time prediction...")

# Extract temporal features
df_processed["dep_hour"] = df_processed["std"].dt.hour
df_processed["dep_day_of_week"] = df_processed["std"].dt.dayofweek
df_processed["dep_month"] = df_processed["std"].dt.month
df_processed["dep_day_of_year"] = df_processed["std"].dt.dayofyear

# Circular encoding for time-based features
print("Creating circular encodings for temporal features...")

# Hour of day (0-23) - circular encoding
df_processed["dep_hour_sin"] = np.sin(2 * np.pi * df_processed["dep_hour"] / 24)
df_processed["dep_hour_cos"] = np.cos(2 * np.pi * df_processed["dep_hour"] / 24)

# Day of week (0-6) - circular encoding
df_processed["dep_day_of_week_sin"] = np.sin(
    2 * np.pi * df_processed["dep_day_of_week"] / 7
)
df_processed["dep_day_of_week_cos"] = np.cos(
    2 * np.pi * df_processed["dep_day_of_week"] / 7
)

# Month (1-12) - circular encoding
df_processed["dep_month_sin"] = np.sin(2 * np.pi * (df_processed["dep_month"] - 1) / 12)
df_processed["dep_month_cos"] = np.cos(2 * np.pi * (df_processed["dep_month"] - 1) / 12)

print("Feature engineering completed!")
print(f"Final processed dataset shape: {df_processed.shape}")

Starting feature engineering for time prediction...
Creating circular encodings for temporal features...
Feature engineering completed!
Final processed dataset shape: (390856, 27)


In [6]:
# Prepare data for H2O Regression
print("Preparing data for H2O regression...")

# Select features for modeling
feature_columns = [
    # Categorical features
    "dep_airport",
    "arr_airport",
    "service_type",
    # Primary predictors (scheduled times)
    "scheduled_dep_minutes",
    "scheduled_arr_minutes",
    "scheduled_duration",
    # Circular encoded temporal features
    "dep_hour_sin",
    "dep_hour_cos",
    "dep_day_of_week_sin",
    "dep_day_of_week_cos",
    "dep_month_sin",
    "dep_month_cos",
    "flight_id",
]

# Regression targets
target_columns = ["dep_delay", "arr_delay"]

print(f"Total features selected: {len(feature_columns)}")
print(f"Regression targets: {target_columns}")

# Create modeling dataset
model_data = df_processed[feature_columns + target_columns].copy()
print(f"Modeling dataset shape: {model_data.shape}")

Preparing data for H2O regression...
Total features selected: 13
Regression targets: ['dep_delay', 'arr_delay']
Modeling dataset shape: (390856, 15)


In [7]:
# Remove duplicates from the entire dataset first
temp_df = model_data.copy()
temp_df["std"] = pd.to_datetime(df_processed["std"])  # Keep std for temporal
temp_df["sta"] = pd.to_datetime(df_processed["sta"])  # Keep sta for temporal


print(f"Original dataset shape: {temp_df.shape}")
temp_df = temp_df.drop_duplicates()
print(f"After removing duplicates: {temp_df.shape}")

# Create test set: October 2024
test_mask = (temp_df["std"].dt.year == 2024) & (temp_df["std"].dt.month == 10)
test_df = temp_df[test_mask].copy()

# Create train+validation set: everything else
train_valid_df = temp_df[~test_mask].copy()

# Shuffle the train+validation data before splitting to avoid temporal bias
train_valid_df = train_valid_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Split train+validation into train and validation (80/20 split of remaining data)
train_size = int(0.99 * len(train_valid_df))
train_df = train_valid_df.iloc[:train_size]
valid_df = train_valid_df.iloc[train_size:]

# Convert to H2O Frame
print("Converting to H2O Frame...")
# Convert back to H2O frames
test = h2o.H2OFrame(test_df)
train = h2o.H2OFrame(train_df)
valid = h2o.H2OFrame(valid_df)

# Convert categorical columns to factors
categorical_cols = ["dep_airport", "arr_airport", "service_type", "season"]
# for col in categorical_cols:
#     if col in h2o_frame.columns:
#         h2o_frame[col] = h2o_frame[col].asfactor()

print("H2O Frame created successfully!")
print(f"H2O Test Frame shape: {test.shape}")
print(f"H2O Train Frame shape: {train.shape}")
print(f"H2O Validation Frame shape: {valid.shape}")

Original dataset shape: (390856, 17)
After removing duplicates: (390856, 17)
Converting to H2O Frame...


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
H2O Frame created successfully!
H2O Test Frame shape: (4628, 17)
H2O Train Frame shape: (382365, 17)
H2O Validation Frame shape: (3863, 17)


In [8]:
# Split data and train regression models
print("Splitting data for regression training...")

print(f"Train set shape: {train.shape}")
print(f"Validation set shape: {valid.shape}")
print(f"Test set shape: {test.shape}")

# Define predictors and targets
x_dep = feature_columns
x_arr = feature_columns

y_dep = "dep_delay"  # First target: departure delay
y_arr = "arr_delay"  # Second target: arrival delay

print(f"Features (x): {len(x_arr)} features")
print(f"Departure target (y_dep): {y_dep}")
print(f"Arrival target (y_arr): {y_arr}")

# Train departure time prediction model
print("\n=== Training Departure delay prediction Model ===")
aml_departure = H2OAutoML(
    max_models=15,  # Increased for more model variety
    max_runtime_secs=900,  # 15 minutes for better models
    seed=42,
    project_name="flight_departure_time_prediction",
    sort_metric="RMSE",
    nfolds=-1,
    # Include GPU-accelerated algorithms
    include_algos=["XGBoost", "DeepLearning", "GBM", "DRF", "GLM"],  # "StackedEnsemble"
    preprocessing=["target_encoding"],
    # Balance exploration vs exploitation
    exploitation_ratio=0.1,
)

# Train departure model
print("Training departure time prediction model...")
aml_departure.train(x=x_dep, y=y_dep, training_frame=train, validation_frame=valid)
print(
    f"Departure model training completed! Leaderboard contains {len(aml_departure.leaderboard)} models"
)

departure_prediction_delay = aml_departure.leader.predict(train)
train["dep_prediction_delay"] = departure_prediction_delay

Splitting data for regression training...
Train set shape: (382365, 17)
Validation set shape: (3863, 17)
Test set shape: (4628, 17)
Features (x): 13 features
Departure target (y_dep): dep_delay
Arrival target (y_arr): arr_delay

=== Training Departure delay prediction Model ===
Training departure time prediction model...
AutoML progress: |
12:21:15.533: User specified a validation frame with cross-validation still enabled. Please note that the models will still be validated using cross-validation only, the validation frame will be used to provide purely informative validation metrics on the trained models.
12:21:15.560: Target Encoding integration in AutoML is in an experimental stage, the models obtained with this feature can not yet be downloaded as MOJO for production.

███████████████████████████████
12:26:24.530: GBM_lr_annealing_selection_AutoML_1_20250926_122115 [GBM lr_annealing] failed: java.lang.NullPointerException

████████████████████████████████| (done) 100%
Departure mod

In [9]:
x = feature_columns
print(x)

# Train arrival time prediction model
print("\n=== Training Arrival Time Prediction Model ===")
aml_arrival = H2OAutoML(
    max_models=15,  # Increased for more model variety
    max_runtime_secs=900,  # 15 minutes for better models
    seed=42,
    project_name="flight_arrival_time_prediction",
    sort_metric="RMSE",
    nfolds=-1,
    # Include GPU-accelerated algorithms
    include_algos=["XGBoost", "DeepLearning", "GBM", "DRF", "GLM"],  # "StackedEnsemble"
    # Enable preprocessing
    preprocessing=["target_encoding"],
    # Balance exploration vs exploitation
    exploitation_ratio=0.1,
)

# Train arrival model
print("Training arrival delay prediction model...")
aml_arrival.train(x=x_arr, y=y_arr, training_frame=train, validation_frame=valid)
print(
    f"Arrival model training completed! Leaderboard contains {len(aml_arrival.leaderboard)} models"
)

print("\n=== Both Models Trained Successfully! ===")

['dep_airport', 'arr_airport', 'service_type', 'scheduled_dep_minutes', 'scheduled_arr_minutes', 'scheduled_duration', 'dep_hour_sin', 'dep_hour_cos', 'dep_day_of_week_sin', 'dep_day_of_week_cos', 'dep_month_sin', 'dep_month_cos', 'flight_id']

=== Training Arrival Time Prediction Model ===
Training arrival delay prediction model...
AutoML progress: |


12:26:27.191: User specified a validation frame with cross-validation still enabled. Please note that the models will still be validated using cross-validation only, the validation frame will be used to provide purely informative validation metrics on the trained models.
12:26:27.192: Target Encoding integration in AutoML is in an experimental stage, the models obtained with this feature can not yet be downloaded as MOJO for production.

███████████████████████████████████████████████████████████████| (done) 100%

12:30:57.761: GBM_lr_annealing_selection_AutoML_2_20250926_122627 [GBM lr_annealing] failed: java.lang.NullPointerException

Arrival model training completed! Leaderboard contains 15 models

=== Both Models Trained Successfully! ===


In [10]:
h2o.save_model(
    aml_departure.get_best_model(),
    "../models/departure_delay_model",
    filename="best",
    force=True,
)
h2o.save_model(
    aml_arrival.get_best_model(),
    "../models/arrival_delay_model",
    filename="best",
    force=True,
)

'/lhome/kristsey/AvinorHackathon/models/arrival_delay_model/best'

In [11]:
dep_predict = h2o.load_model("../models/departure_delay_model/best")
arr_predict = h2o.load_model("../models/arrival_delay_model/best")

In [14]:
prediction_df = test[x_arr]
print(prediction_df.columns)
print(train.columns)


predicted_departure_delay_october = dep_predict.predict(prediction_df)
predicted_arrival_delay_october = arr_predict.predict(prediction_df)
predicted_departure_delay_october = (
    predicted_departure_delay_october.as_data_frame().values
)

prediction_df = test.as_data_frame()[
    ["flight_id", "dep_airport", "arr_airport", "service_type", "std", "sta"]
]


predicted_arrival_delay_october = predicted_arrival_delay_october.as_data_frame()
prediction_df["predicted_departure_delay"] = predicted_departure_delay_october
prediction_df["predicted_arrival_delay"] = predicted_arrival_delay_october
prediction_df.to_csv("../data/predictions/predicted_schedule_october.csv", index=False)

['dep_airport', 'arr_airport', 'service_type', 'scheduled_dep_minutes', 'scheduled_arr_minutes', 'scheduled_duration', 'dep_hour_sin', 'dep_hour_cos', 'dep_day_of_week_sin', 'dep_day_of_week_cos', 'dep_month_sin', 'dep_month_cos', 'flight_id']
['dep_airport', 'arr_airport', 'service_type', 'scheduled_dep_minutes', 'scheduled_arr_minutes', 'scheduled_duration', 'dep_hour_sin', 'dep_hour_cos', 'dep_day_of_week_sin', 'dep_day_of_week_cos', 'dep_month_sin', 'dep_month_cos', 'flight_id', 'dep_delay', 'arr_delay', 'std', 'sta', 'dep_prediction_delay']
gbm prediction progress: |███████████████████████████████████████████████████████| (done) 100%
gbm prediction progress: |███████████████████████████████████████████████████████| (done) 100%


In [ ]:
# Evaluate Regression Models
print("=== EVALUATING REGRESSION MODELS ===")

# Get best models
best_dep_model = aml_departure.leader
best_arr_model = aml_arrival.leader

print(f"Best departure model: {type(best_dep_model).__name__}")
print(f"Best arrival model: {type(best_arr_model).__name__}")

# Evaluate on test set
print("\n=== DEPARTURE TIME PREDICTION PERFORMANCE ===")
dep_performance = best_dep_model.model_performance(test)
print(f"RMSE: {dep_performance.rmse():.2f} minutes")
print(f"MAE: {dep_performance.mae():.2f} minutes")
print(f"R²: {dep_performance.r2():.4f}")

print("\n=== ARRIVAL TIME PREDICTION PERFORMANCE ===")
arr_performance = best_arr_model.model_performance(test)
print(f"RMSE: {arr_performance.rmse():.2f} minutes")
print(f"MAE: {arr_performance.mae():.2f} minutes")
print(f"R²: {arr_performance.r2():.4f}")

# Display leaderboards
print("\n=== DEPARTURE TIME PREDICTION LEADERBOARD ===")
print(aml_departure.leaderboard.as_data_frame())

print("\n=== ARRIVAL TIME PREDICTION LEADERBOARD ===")
print(aml_arrival.leaderboard.as_data_frame())

# BEFORE DEPARTURE_PREDICTION FEATURE
# Best departure model: H2OStackedEnsembleEstimator
# Best arrival model: H2OGradientBoostingEstimator

# === DEPARTURE TIME PREDICTION PERFORMANCE ===
# RMSE: 16.88 minutes
# MAE: 9.28 minutes
# R²: 0.0093

# === ARRIVAL TIME PREDICTION PERFORMANCE ===
# RMSE: 26.22 minutes
# MAE: 11.07 minutes
# R²: 0.0746

=== EVALUATING REGRESSION MODELS ===
Best departure model: H2OStackedEnsembleEstimator
Best arrival model: H2OStackedEnsembleEstimator

=== DEPARTURE TIME PREDICTION PERFORMANCE ===
RMSE: 16.70 minutes
MAE: 9.17 minutes
R²: 0.0304

=== ARRIVAL TIME PREDICTION PERFORMANCE ===
RMSE: 27.29 minutes
MAE: 11.96 minutes
R²: -0.0024

=== DEPARTURE TIME PREDICTION LEADERBOARD ===
                                             model_id       rmse         mse  \
0   StackedEnsemble_AllModels_1_AutoML_1_20250926_...  26.465223  700.408034   
1   StackedEnsemble_BestOfFamily_1_AutoML_1_202509...  26.562204  705.550673   
2                      GBM_1_AutoML_1_20250926_102206  26.663204  710.926431   
3                      GBM_3_AutoML_1_20250926_102206  26.699495  712.863049   
4                      GBM_4_AutoML_1_20250926_102206  26.728664  714.421470   
5                      GBM_5_AutoML_1_20250926_102206  26.737827  714.911414   
6         GBM_grid_1_AutoML_1_20250926_102206_model_1  26.738303  

In [ ]:
# Plot variable importance

dep_varimp = best_dep_model.varimp(use_pandas=True)
arr_varimp = best_arr_model.varimp(use_pandas=True)
plt.figure(figsize=(10, 8))
if dep_varimp and len(dep_varimp) > 0:
    top_features = dep_varimp.head(15)
    plt.barh(range(len(top_features)), top_features["percentage"])
    plt.yticks(range(len(top_features)), top_features["variable"])
    plt.xlabel("Relative Importance (%)")
    plt.title("Top 15 Feature Importance")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("No variable importance data available")

No variable importance data available


<Figure size 1000x800 with 0 Axes>

In [ ]:
# Plot variable importance
plt.figure(figsize=(10, 8))
if arr_varimp and len(arr_varimp) > 0:
    top_features = arr_varimp.head(15)
    plt.barh(range(len(top_features)), top_features["percentage"])
    plt.yticks(range(len(top_features)), top_features["variable"])
    plt.xlabel("Relative Importance (%)")
    plt.title("Top 15 Feature Importance")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("No variable importance data available")

No variable importance data available


<Figure size 1000x800 with 0 Axes>